# Batch process many files

**The job.** Forty files. Read each one, pull out the numbers, add them up.

Until now every job in this series handled one thing. A step ran once. That is
fine for one page or one table and useless for a folder.

A `map` step runs its node once per item and gives back a list. That is the
whole difference, and it is worth seeing because it changes what the graph can
say, not just how it runs.

**In:** 40 small text files, two of them broken.
**Out:** a total, and a list of the files that could not be read.
**Files:** a summary, and a list of the failures.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

def node(node_id, capability, takes, gives, **kw):
    """Describe one node. Ports are (name, type) pairs."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives), **kw)

def stage(sid, name, takes, gives, capability, candidates):
    """Describe one step of the job, and what could do it."""
    return StageDefinition(
        id=sid, name=name, required_capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives),
        success=f"{name} produced its declared output",
        candidates=tuple(candidates))

def build(title, task, stages, nodes, edges=()):
    """Put it together and check it before anything runs."""
    bench = WorkbenchDefinition(
        title=title, task=task, stages=tuple(stages), nodes=tuple(nodes),
        edges=tuple(edges),
        candidates=tuple(NodeCandidate(id=n.id, node_id=n.id) for n in nodes))
    problems = bench.validate()
    print("problems:", problems if problems else "none")
    return bench

print("ready")

ready


## The input

Forty readings files. Two are corrupt, because forty real files always include
a couple that are.

In [2]:
import random
rng = random.Random(5)

folder = WORK / "readings"
folder.mkdir()
for i in range(40):
    rows = [f"{rng.randint(1, 500)}" for _ in range(rng.randint(3, 8))]
    if i in (11, 29):
        rows[1] = "n/a"                     # the two broken ones
    (folder / f"day-{i:02d}.txt").write_text("\n".join(rows))

print(f"{len(list(folder.glob('*.txt')))} files")
print("day-00:", (folder / "day-00.txt").read_text().replace("\n", " "))
print("day-11:", (folder / "day-11.txt").read_text().replace("\n", " "), " <- broken")

40 files
day-00: 131 380 184 408 354 483 431
day-11: 277 n/a 348 321  <- broken


## The steps

Four. The middle one is the new part: `kind="map"`.

Note the port types. The list step gives `List[Path]`. The map step consumes
`List[Path]` and produces `List[Reading]` — but the **node** inside it takes one
`Path` and gives one `Reading`. The stage talks about the collection, the node
talks about one item, and that is what `map` means.

In [3]:
nodes = [
    node("list.files",  "list",    [],                       [("out", "List[Path]")]),
    node("read.one",    "read",    [("in", "Path")],         [("out", "Reading")]),
    node("split.ok",    "split",   [("in", "List[Reading]")],[("ok", "List[Reading]"), ("bad", "List[Reading]")]),
    node("total.all",   "total",   [("in", "List[Reading]")],[("out", "Summary")]),
]

stages = [
    stage("list",  "List the files", [],                        [("out", "List[Path]")],    "list",  ["list.files"]),
    StageDefinition(id="read", name="Read each one", kind="map",
                    required_capabilities=("read",),
                    inputs=(PortSpec("in", "List[Path]"),),
                    outputs=(PortSpec("out", "List[Reading]"),),
                    success="every file was attempted",
                    candidates=("read.one",)),
    stage("split", "Good from bad",  [("in", "List[Reading]")], [("ok", "List[Reading]"), ("bad", "List[Reading]")], "split", ["split.ok"]),
    stage("total", "Add them up",    [("in", "List[Reading]")], [("out", "Summary")],       "total", ["total.all"]),
]

edges = [Edge("list", "read"), Edge("read", "split"),
         Edge("split", "total", from_port="ok")]

bench = build("Process a folder of files",
              "Read forty files, add up the numbers, report the broken ones.",
              stages, nodes, edges)
print("kinds:", {s.id: s.kind for s in bench.leaf_stages})

problems: none
kinds: {'list': 'atomic', 'read': 'map', 'split': 'atomic', 'total': 'atomic'}


In [4]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 226" width="1100" height="226" style="max-width:none" role="img"><defs><marker id="bg43508761-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">List the files</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="398.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="310" y="79.0" width="186" height="52" rx="7" fill="none" stroke="#8a93a0" stroke-width="1" opacity=".45"/><rect x="305" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="314" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Read each one</text><text x="482" y="93.0" text-anchor="end" font-size="9" font-weight="700" fill="#2d6cb5">MAP</text><text x="314" y="108.0" font-size="9.5" fill="#68737f">1 candidate · per item</text></g><text x="643.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="550" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="559" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Good from bad</text><text x="559" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="888.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="795" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="804" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Add them up</text><text x="804" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C275.5,100.0 275.5,100.0 305,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg43508761-arrow)"/><path d="M491,100.0 C520.5,100.0 520.5,100.0 550,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg43508761-arrow)"/><path d="M736,100.0 C765.5,100.0 765.5,100.0 795,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg43508761-arrow)"/><text x="765.5" y="95.0" text-anchor="middle" font-size="9" fill="#68737f">ok</text></svg>', title='Process a folder of files — shape', note='a chain. A doubled outline runs once per item. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=226)

## The code

`read_one` handles **one** file. It never sees the list. That is the point: the
node stays simple and the graph handles the "for each".

In [5]:
def list_files():
    return sorted(folder.glob("*.txt"))

def read_one(**kw):
    """One file in, one reading out. No loop anywhere in here."""
    path = kw["in"]
    numbers, bad = [], []
    for line in path.read_text().splitlines():
        try:
            numbers.append(int(line))
        except ValueError:
            bad.append(line)
    return {"file": path.name, "values": numbers, "unreadable": bad}

def split_ok(**kw):
    readings = kw["in"]
    return {"ok": [r for r in readings if not r["unreadable"]],
            "bad": [r for r in readings if r["unreadable"]]}

def total_all(workspace, **kw):
    readings = kw["in"]
    total = sum(v for r in readings for v in r["values"])
    summary = {"files": len(readings), "readings": sum(len(r["values"]) for r in readings),
               "total": total}
    (workspace / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

runtime = execute.Runtime({"list.files": list_files, "read.one": read_one,
                           "split.ok": split_ok, "total.all": total_all})

plan = compile_route(bench, {s.id: s.candidates[0] for s in bench.leaf_stages})
run = execute.run(plan, runtime, workspace=WORK)
print(run.text())

plan plan:93202263e0f97aae7a06d…
4 steps in 0.005s — ok
  ok   list             0.000s  list.files
  ok   read             0.002s  read.one
  ok   split            0.000s  split.ok
  ok   total            0.000s  total.all
  file /home/username/code_projects/repos/browsergraph/notebooks/work/summary.json  54 bytes  sha256:3169b3f7a…


## What came out

In [6]:
ok = run.values[("split", "ok")]
bad = run.values[("split", "bad")]
summary = run.output("total")

print(f"read {len(ok) + len(bad)} files")
print(f"  usable: {len(ok)}")
print(f"  broken: {len(bad)}  -> {[r['file'] for r in bad]}")
print(f"\n{summary['readings']:,} readings, total {summary['total']:,}")

for r in bad:
    print(f"\n{r['file']} could not read: {r['unreadable']}")

read 40 files
  usable: 38
  broken: 2  -> ['day-11.txt', 'day-29.txt']

194 readings, total 48,627

day-11.txt could not read: ['n/a']

day-29.txt could not read: ['n/a']


## When one file blows up

`read_one` above never raises — it collects the bad lines instead. If a node
does raise, the map step names the item, not just the batch.

In [7]:
def read_strict(**kw):
    path = kw["in"]
    return {"file": path.name,
            "values": [int(line) for line in path.read_text().splitlines()],
            "unreadable": []}

strict_run = execute.run(plan, execute.Runtime(
    dict(runtime._functions, **{"read.one": read_strict})), workspace=WORK)

print("ok:", strict_run.ok)
print(next(s for s in strict_run.steps if s.stage == "read").error)

ok: False
read.one: RuntimeError: item 11: invalid literal for int() with base 10: 'n/a'


`item 11` — the eleventh file, which is `day-11.txt`. "The batch failed" would
have left you to find that yourself across forty files.

In [8]:
print("files written:")
for art in run.artifacts:
    print(f"  {art.path:<34} {art.bytes:>8,} bytes  {art.digest[:18]}…")

files written:
  /home/username/code_projects/repos/browsergraph/notebooks/work/summary.json       54 bytes  sha256:3169b3f7a6f…
